In [0]:
# Set up Google Cloud service account keys 
def setup_gcp_creds(credentials_path: str):
    if os.path.exists(credentials_path):
        try:
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"]= credentials_path
            logger.info(f"Found service account file")
            client = storage.Client()
            buckets = list(client.list_buckets())
            logger.info(f"Successfully authenticated. Found {len(buckets)} buckets.")
        except Exception as e:
            logger.error(f"File credential file exists but authentection failed")
            logger.error(f"Details: {e}")
    else:
        print(f"Service account file NOT FOUND at: {credentials_path}")
        dbutils.notebook.exit("File not found")


In [0]:

def gcs_data_ingestion(gcs_bucket_name: str, blob_file_path: str, dbfs_path: str) -> None:
    try:
        # 1. Setup GCS client
        client = storage.Client()


        bucket = client.get_bucket(gcs_bucket_name)
        logger.info(f"GCS bucket: {bucket}")
        blob = bucket.blob(gcs_weather_file_path)
        logger.info(f"Blob in GCS bucket: {blob}")

        # 2. Path to Unity Catalog Volume
        os.makedirs(dbfs_chicago_weather_path, exist_ok=True)
        output_filename = gcs_weather_file_path.split('/')[-1]
        logger.info(f"Output filename: {output_filename}")
        destination_path = f"{dbfs_chicago_weather_path}/{output_filename}"
        logger.info(f"Destination Path: {destination_path}")

        # 3. Write to Unity Catalog Volume

        with open(destination_path, "wb") as f:
            blob.download_to_file(f)

        logger.info(f"File with holiday data ingestested to {destination_path}")
    except Exception as e:
        logger.error(f"File ingestestion failed: {e}")
        dbutils.notebook.exit(f"File ingestestion failed: {e}")